# 🔧 Module 8 — Capstone Preparation: System Integration & Production Patterns
**Duration:** ~3 hours  
This module bridges Module 7 → Capstone. You will build and test each subsystem **in isolation** before wiring them together in the Capstone notebook.

---
| Section | Topic |
|---------|-------|
| 08a | Data ingestion & context window budget |
| 08b | Intelligence core wiring (RAG + ML) |
| 08c | Evaluation harness & cost tracker |

> **Tip:** Complete all cells here before opening the Capstone notebook. Each section ends with an integration test that confirms the subsystem is production-ready.


In [ ]:
# ── Shared imports (run this first) ──────────────────────────────
import os, json, time, datetime, re
from pathlib import Path
from typing import Optional, List
from pydantic import BaseModel, Field, field_validator
from datasets import load_dataset
import pandas as pd
from openai import OpenAI
from anthropic import Anthropic
from llm_checker import check, hint, evaluate, progress, show_exercise

openai_client = OpenAI()
claude_client = Anthropic()
lm_client     = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

Path("data/capstone").mkdir(parents=True, exist_ok=True)
print("✅  Module 8 imports ready")

---
## 08a — Data Ingestion & Context Window Budget

**Goal:** Build a reliable ingestion pipeline that enforces a 2000-token context window per customer, achieves ≥ 90% Pydantic pass rate, and logs any failures.


In [ ]:
# ── Load 100 CFPB complaints ──────────────────────────────────────
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

def trim_to_budget(text: str, budget: int = 1800) -> str:
    """Hard-trim text so it fits within budget tokens."""
    tokens = enc.encode(text)
    if len(tokens) <= budget:
        return text
    return enc.decode(tokens[:budget])

ds = load_dataset("cfpb/us-consumer-finance-complaints", split="train").select(range(100))
complaints_df = pd.DataFrame(ds).fillna("")
print(f"Loaded {len(complaints_df)} complaints | columns: {list(complaints_df.columns)}")

In [ ]:
# ── Pydantic schema for complaint records ─────────────────────────

class ComplaintRecord(BaseModel):
    customer_id:       str
    category:          str   = Field(description="One of: credit_card, mortgage, checking_account, loan, other")
    sentiment:         str   = Field(description="positive | neutral | negative")
    urgency:           str   = Field(description="low | medium | high")
    product_mentioned: str
    action_required:   bool
    summary:           str   = Field(max_length=200)

    @field_validator("category")
    @classmethod
    def valid_category(cls, v):
        allowed = {"credit_card","mortgage","checking_account","loan","other"}
        if v not in allowed:
            raise ValueError(f"Invalid category: {v}")
        return v

def extract_complaint_record(row: pd.Series) -> Optional[ComplaintRecord]:
    """Extract structured ComplaintRecord from a complaint row."""
    complaint_text = trim_to_budget(
        row.get("complaint_what_happened", "") or row.get("consumer_complaint_narrative", ""), 1800
    )
    if not complaint_text.strip():
        return None

    prompt = f"""Extract structured data from this CFPB complaint. Reply ONLY with valid JSON.

COMPLAINT: {complaint_text}

JSON schema:
{{
  "category": "credit_card|mortgage|checking_account|loan|other",
  "sentiment": "positive|neutral|negative",
  "urgency": "low|medium|high",
  "product_mentioned": "string (product name from complaint)",
  "action_required": true/false,
  "summary": "one sentence ≤ 200 chars"
}}"""

    try:
        msg = lm_client.chat.completions.create(
            model="local-model", max_tokens=300,
            messages=[{"role":"user","content": prompt}]
        )
        raw = msg.choices[0].message.content.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
        data = json.loads(raw)
        return ComplaintRecord(customer_id=str(row.name), **data)
    except Exception as e:
        return None

# Quick smoke test
rec = extract_complaint_record(complaints_df.iloc[0])
print("Sample record:", rec)

---
### 🟡 EXERCISE Ex 08a-1 — Ingestion Pipeline with Pass Rate Logging

Run the full ingestion pipeline on 100 complaints. Log:
- Pydantic pass rate (must be ≥ 90%)
- Context window violations (must be 0 — trim_to_budget prevents them)
- Failed parses with error messages

**Hints:**
- Use `extract_complaint_record()` on each row
- Track successes vs failures in a dict
- Save results to `data/capstone/complaints_extracted.jsonl`


In [ ]:
show_exercise(
    "08a-1", "Ingestion Pipeline with Pass Rate Logging",
    "Run extract_complaint_record on 100 CFPB complaints. Track pass rate and failures. "
    "Save successful records to complaints_extracted.jsonl.",
    hints=[
        "▸ Use extract_complaint_record() in a loop",
        "▸ Track successes and failures separately",
        "▸ Save with: json.dumps(rec.model_dump()) + '\\n'",
    ],
    checks=[
        "100 complaints processed",
        "Pydantic pass rate ≥ 90% logged",
        "data/capstone/complaints_extracted.jsonl written",
        "No context window exceeds 2000 tokens"
    ]
)

In [ ]:
# ── YOUR SOLUTION — Ex 08a-1 ──────────────────────────────────────
records     = []
failures    = []
token_violations = 0

for i, row in complaints_df.iterrows():
    # Check token budget
    text = row.get("complaint_what_happened", "") or row.get("consumer_complaint_narrative", "")
    trimmed = trim_to_budget(text)
    if count_tokens(trimmed) > 2000:
        token_violations += 1

    # YOUR CODE HERE — extract and append
    # rec = extract_complaint_record(row)
    # ...

pass_rate = len(records) / max(len(records) + len(failures), 1)
print(f"Pass rate: {pass_rate:.1%} | Failures: {len(failures)} | Token violations: {token_violations}")

# Save to JSONL
out_path = Path("data/capstone/complaints_extracted.jsonl")
with open(out_path, "w") as f:
    for rec in records:
        f.write(json.dumps(rec.model_dump()) + "\n")
print(f"✅ Saved {len(records)} records → {out_path}")

In [ ]:
# ── Auto-check Ex 08a-1 ───────────────────────────────────────────
out_path = Path("data/capstone/complaints_extracted.jsonl")
n_saved  = sum(1 for _ in out_path.open()) if out_path.exists() else 0

check([
    (len(records) + len(failures) == 100, "All 100 complaints processed"),
    (pass_rate >= 0.90,                   "Pydantic pass rate ≥ 90%"),
    (token_violations == 0,               "Zero context-window violations"),
    (out_path.exists(),                   "complaints_extracted.jsonl written"),
    (n_saved >= 90,                       "≥ 90 records saved to JSONL"),
], exercise_id="08a-1")

---
## ✅ Lesson 08a Readiness
- ☐ 100 complaints ingested into `complaints.jsonl`
- ☐ Pydantic pass rate ≥ 90% logged
- ☐ Context window never exceeds 2000 tokens